<div style="font-size:30px;font-weight:700;color:#111827;padding-bottom:8px;margin:18px 0;">
Tool Calling을 지원하지 않는 모델로 도구 사용하기 (LangChain)
</div>

`bind_tools()` 같은 **네이티브 Tool Calling 기능이 없는 소형 모델**도,
프롬프트와 LangChain 체인 구성만으로 도구를 사용하게 만들 수 있습니다.

이 노트북은 `105_langchain` 노트북의 모델(`unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit`)과
`QwenChatModel` 클래스를 그대로 사용하고, 전체 흐름을 **LCEL(LangChain Expression Language) 체인 하나**로 구성합니다.

| 단계 | 역할 | LangChain 구성 요소 |
|---|---|---|
| ① 도구 선택 | 질문을 보고 어떤 도구를 쓸지 JSON으로 출력 | `ChatPromptTemplate` → `llm` → `JsonOutputParser` |
| ② 도구 실행 | 선택 결과를 받아 파이썬 함수 호출 | `RunnableLambda` |
| ③ 최종 답변 | 도구 결과(observation)를 근거로 자연어 답변 | `ChatPromptTemplate` → `llm` → `StrOutputParser` |

세 단계를 `RunnablePassthrough.assign()`으로 이어 붙이면 `chain.invoke({"input": 질문})` 한 번으로 끝납니다.

# 기본 환경 설정

In [ ]:
# %%capture
# %pip install -q -U unsloth langchain-core grandalf

# GPU와 실행 환경 확인

Unsloth를 다른 Transformers 관련 라이브러리보다 먼저 가져오는 것이 좋습니다.

In [ ]:
from unsloth import FastLanguageModel
import torch

In [ ]:
print(f"PyTorch 버전: {torch.__version__}")
print(f"사용 GPU: {torch.cuda.get_device_name(0)}")

# 4비트 Qwen2.5 모델 로딩

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
)

model.eval()

# 커스텀 ChatModel 클래스 작성

`105_langchain` 노트북의 `QwenChatModel`을 그대로 사용합니다.

> 한 가지만 다릅니다: `max_length` 대신 **`max_new_tokens`**를 사용합니다.
> `max_length`는 *프롬프트 길이를 포함한* 전체 길이 제한이라, 도구 스키마가 들어간 긴 시스템 프롬프트를 쓰면
> 답변이 나오기도 전에 잘릴 수 있습니다. `max_new_tokens`는 *새로 생성할 토큰 수*만 제한합니다.

In [ ]:
import torch
from threading import Thread
from typing import Any, Iterator, List, Optional
from pydantic import ConfigDict

In [ ]:
from langchain_core.callbacks import CallbackManagerForLLMRun
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, AIMessageChunk, BaseMessage, HumanMessage, SystemMessage
from langchain_core.outputs import ChatGeneration, ChatGenerationChunk, ChatResult

from transformers import TextIteratorStreamer

In [ ]:
class QwenChatModel(BaseChatModel):
    """Qwen2.5-Instruct를 LangChain ChatModel로 감싸는 클래스."""

    # BaseChatModel은 Pydantic 기반이므로 필드 선언이 필요합니다.
    model: Any
    tokenizer: Any

    max_tokens: int = 512
    do_sample: bool = True
    temperature: float = 0.7
    top_p: float = 0.9

    model_config = ConfigDict(arbitrary_types_allowed=True)

    @property
    def _llm_type(self) -> str:
        return "qwen2.5-custom-chatmodel"

    def _tokenize(self, messages: List[BaseMessage]):
        """LangChain 메시지를 Qwen 채팅 형식으로 변환하고 토큰화합니다."""
        chat = []

        for message in messages:
            if isinstance(message, SystemMessage):
                role = "system"
            elif isinstance(message, HumanMessage):
                role = "user"
            elif isinstance(message, AIMessage):
                role = "assistant"
            else:
                role = "user"

            chat.append({"role": role, "content": message.content})

        inputs = self.tokenizer.apply_chat_template(
            chat,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )
        return inputs.to(self.model.device)

    def _generation_options(self):
        """model.generate()에 공통으로 전달할 옵션입니다."""
        options = {
            "max_length": self.max_tokens,
            "do_sample": self.do_sample,
            "pad_token_id": self.tokenizer.pad_token_id,
        }

        if self.do_sample:
            options["temperature"] = self.temperature
            options["top_p"] = self.top_p

        return options

    def _generate(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> ChatResult:
        """invoke()와 batch()가 사용하는 메서드입니다."""
        inputs = self._tokenize(messages)
        input_length = inputs["input_ids"].shape[1]

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                **self._generation_options(),
            )

        new_tokens = outputs[0][input_length:]
        text = self.tokenizer.decode(
            new_tokens,
            skip_special_tokens=True,
        ).strip()

        return ChatResult(
            generations=[ChatGeneration(message=AIMessage(content=text))]
        )

    def _stream(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> Iterator[ChatGenerationChunk]:
        """stream()이 사용하는 메서드입니다."""
        inputs = self._tokenize(messages)

        streamer = TextIteratorStreamer(
            self.tokenizer,
            skip_prompt=True,
            skip_special_tokens=True,
        )

        thread = Thread(
            target=self.model.generate,
            kwargs={
                **inputs,
                **self._generation_options(),
                "streamer": streamer,
            },
        )
        thread.start()

        for text in streamer:
            chunk = ChatGenerationChunk(
                message=AIMessageChunk(content=text)
            )

            if run_manager:
                run_manager.on_llm_new_token(text, chunk=chunk)

            yield chunk

        thread.join()

같은 모델을 두 가지 설정으로 감쌉니다.

- `selector_llm` : **도구 선택용**. JSON을 정확히 뽑아야 하므로 `do_sample=False`(항상 같은 답)
- `answer_llm` : **최종 답변용**. 자연스러운 문장을 위해 샘플링 사용

In [ ]:
selector_llm = QwenChatModel(model=model, tokenizer=tokenizer, max_tokens=1024, do_sample=False)
answer_llm   = QwenChatModel(model=model, tokenizer=tokenizer, max_tokens=2048)

In [ ]:
# 모델이 정상 동작하는지 간단히 확인
print(answer_llm.invoke("안녕? 한 문장으로 자기소개해 줘.").content)

# 툴 정의

실제 서비스에서는 API 호출로 바꾸면 됩니다. 여기서는 동작 확인용 데모 함수 2개를 만듭니다.

In [ ]:
def get_weather(city: str) -> str:
    """도시의 현재 날씨를 조회 (데모)"""
    return f"{city}: 맑음, 25℃ (데모)"


def get_exchange_rate(currency: str) -> str:
    """통화의 원화 환율을 조회 (데모)"""
    rates = {"USD": 1380, "JPY": 9.2, "EUR": 1490}
    if currency.upper() not in rates:
        return f"{currency}: 환율 정보 없음"
    return f"1 {currency.upper()} = {rates[currency.upper()]} KRW (데모)"

모델에게 보여줄 **도구 스키마**와, 실제로 실행할 **파이썬 함수**를 이름으로 연결합니다.

In [ ]:
TOOLS = {
    "get_weather": {
        "description": "도시의 현재 날씨를 조회",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string", "description": "도시 이름"}},
            "required": ["city"],
        },
    },
    "get_exchange_rate": {
        "description": "통화의 원화(KRW) 환율을 조회",
        "parameters": {
            "type": "object",
            "properties": {"currency": {"type": "string", "description": "통화 코드 (USD, JPY, EUR)"}},
            "required": ["currency"],
        },
    },
}

TOOL_FUNCS = {
    "get_weather": get_weather,
    "get_exchange_rate": get_exchange_rate,
}

# ① 도구 선택 체인 — `prompt | llm | parser`

모델에게 "질문에 맞는 도구와 인자를 JSON으로만 출력하라"고 지시합니다.
도구가 필요 없는 질문이면 `"tool": "none"`을 출력하게 합니다.

> 프롬프트 템플릿에서 `{` `}`는 변수 자리표시자로 해석되므로, JSON 예시는 `{{` `}}`로 씁니다.

In [ ]:
import json
import re

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

In [ ]:
select_instruct = """\
너는 도구를 선택해 답을 찾는 어시스턴트다.
반드시 아래 JSON 형식만 출력하라. 설명/문장/코드블록 금지.

출력 형식:
{{"tool": "<도구 이름 또는 none>", "args": {{...}}}}

규칙:
- 사용 가능한 도구 이름: {tool_names}, none
- args는 선택한 도구의 parameters와 정확히 일치해야 한다.
- 도구가 필요 없는 일반 질문이면 {{"tool": "none", "args": {{}}}} 를 출력한다.

사용 가능 도구 정의:
{tool_schema}
""".strip()

In [ ]:
select_prompt = ChatPromptTemplate.from_messages([
    ("system", select_instruct),
    ("human", "사용자 질문: {input}"),
]).partial(
    tool_names=", ".join(TOOLS.keys()),
    tool_schema=json.dumps(TOOLS, ensure_ascii=False, indent=2),
)

# 실제로 모델에 들어가는 프롬프트 확인
print(select_prompt.format(input="서울 날씨 알려줘"))

소형 모델은 가끔 JSON 앞뒤에 문장이나 코드블록을 붙입니다.
`JsonOutputParser`가 실패하면 첫 번째 `{ ... }` 블록만 잘라 다시 파싱하는 **안전장치**를 둡니다.

In [ ]:
json_parser = JsonOutputParser()


def safe_parse(message) -> dict:
    """JSON 파싱 실패 시 텍스트에서 첫 { ... } 블록을 찾아 다시 시도합니다."""
    text = message.content if hasattr(message, "content") else str(message)
    try:
        return json_parser.parse(text)
    except Exception:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except Exception:
                pass
        return {"tool": "none", "args": {}, "raw": text}

In [ ]:
select_chain = select_prompt | selector_llm | RunnableLambda(safe_parse)

In [ ]:
# 도구 선택 단계만 따로 실행해 보기
for q in ["서울 날씨 알려줘", "달러 환율 얼마야?", "파이썬이 뭐야?"]:
    print(q, "→", select_chain.invoke({"input": q}))

# ② 도구 실행 — `RunnableLambda`

선택 결과(dict)를 받아 실제 파이썬 함수를 호출하고, 그 결과를 **observation**으로 돌려줍니다.

In [ ]:
def run_tool(selection: dict) -> dict:
    tool = selection.get("tool", "none")

    # 도구를 선택하지 않았거나, 없는 도구 이름이면 실행하지 않음
    if tool == "none" or tool not in TOOL_FUNCS:
        return {"tool": "none", "args": {}, "observation": None}

    args = selection.get("args", {}) or {}

    try:
        result = TOOL_FUNCS[tool](**args)
    except Exception as e:
        result = f"TOOL_ERROR: {e}"

    return {"tool": tool, "args": args, "observation": result}

In [ ]:
route = RunnableLambda(run_tool)

# 선택 체인과 이어서 확인
(select_chain | route).invoke({"input": "서울 날씨 알려줘"})

# ③ 최종 답변 체인 — `prompt | llm | StrOutputParser`

도구 결과가 있으면 그것을 근거로, 없으면 모델이 아는 대로 답하게 합니다.

In [ ]:
final_instruct = """\
너는 도구 결과를 참고하여 간결하고 정확하게 한국어로 답한다.
도구결과: {observation}

지시:
- 도구결과가 있으면 그것을 근거로 한 문단의 최종 답을 써라.
- 도구결과가 None이면 도구 없이 바로 답하되, 모르면 솔직히 모른다고 말해라.
""".strip()

In [ ]:
final_prompt = ChatPromptTemplate.from_messages([
    ("system", final_instruct),
    ("human", "사용자 질문: {input}"),
])

final_chain = final_prompt | answer_llm | StrOutputParser()

# 세 단계를 하나의 LCEL 체인으로 연결

`RunnablePassthrough.assign()`은 **입력 dict를 그대로 유지하면서 새 키를 추가**합니다.
각 단계의 결과가 dict에 차곡차곡 쌓이므로, 마지막 프롬프트에서 `input`과 `observation`을 모두 쓸 수 있습니다.

```
{"input": 질문}
  → assign(selection = select_chain)        # {"input", "selection"}
  → assign(routed    = run_tool(selection))  # {"input", "selection", "routed"}
  → assign(observation = routed.observation) # {"input", ..., "observation"}
  → final_chain                              # 문자열 답변
```

In [ ]:
tool_chain = (
    RunnablePassthrough.assign(selection=select_chain)
    | RunnablePassthrough.assign(routed=lambda x: run_tool(x["selection"]))
    | RunnablePassthrough.assign(observation=lambda x: x["routed"]["observation"])
    | final_chain
)

In [ ]:
# 체인 구조 시각화
tool_chain.get_graph().print_ascii()

# 엔드-투-엔드 실행

In [ ]:
print(tool_chain.invoke({"input": "서울 날씨 알려줘"}))

In [ ]:
print(tool_chain.invoke({"input": "엔화 환율 알려줘"}))

In [ ]:
# 도구가 필요 없는 질문 → tool: none → 모델이 직접 답변
print(tool_chain.invoke({"input": "LangChain이 뭐야? 한 문장으로."}))

# 중간 과정까지 함께 보기

디버깅이나 교육용으로는 어떤 도구가 선택되었고 어떤 결과가 나왔는지 함께 보는 것이 좋습니다.
마지막 단계에서 `final_chain`의 답변을 `answer` 키로 추가하면 전체 dict를 돌려받을 수 있습니다.

In [ ]:
debug_chain = (
    RunnablePassthrough.assign(selection=select_chain)
    | RunnablePassthrough.assign(routed=lambda x: run_tool(x["selection"]))
    | RunnablePassthrough.assign(observation=lambda x: x["routed"]["observation"])
    | RunnablePassthrough.assign(answer=final_chain)
)

In [ ]:
result = debug_chain.invoke({"input": "부산 날씨 어때?"})

print("선택된 도구 :", result["routed"]["tool"], result["routed"]["args"])
print("도구 결과   :", result["observation"])
print("최종 답변   :", result["answer"])

# 여러 질문 한 번에 처리 — `batch`

로컬 GPU 1개이므로 `max_concurrency=1`로 순차 실행합니다.

In [ ]:
questions = [
    {"input": "대구 날씨 알려줘"},
    {"input": "유로 환율은?"},
    {"input": "김밥 주재료 3가지만 알려줘"},
]

answers = tool_chain.batch(questions, config={"max_concurrency": 1})

for q, a in zip(questions, answers):
    print(f"Q: {q['input']}")
    print(f"A: {a}")
    print("-" * 60)

# 정리

- 네이티브 Tool Calling이 없어도 **프롬프트(JSON 출력 강제) + 파서 + 라우팅 함수**로 도구 사용을 구현할 수 있습니다.
- LangChain에서는 이 흐름을 `prompt | llm | parser`, `RunnableLambda`, `RunnablePassthrough.assign`으로 선언적으로 연결합니다.
- 0.5B급 소형 모델은 JSON 형식을 어길 수 있으므로 `do_sample=False`와 파싱 안전장치가 중요합니다.
- 도구를 추가하려면 `TOOLS`에 스키마를, `TOOL_FUNCS`에 함수를 한 줄씩 등록하면 됩니다.